<a href="https://colab.research.google.com/github/Aatka-Saleem/ML-Core-Implementations/blob/main/06-Deep-Learning-%26--Neural-Networks/Implementing_a_Convolutional_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import keras
from keras.datasets import cifar10
from keras.models import Sequential
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.callbacks import ModelCheckpoint
from sklearn.metrics import confusion_matrix

The following codes can be run in two ways:
1. Run on your local machine using anaconda.
   - To install TensorFlow2 and Keras on Anaconda, run the following commands on the shell:\
     conda install –c conda-forge tensorflow\
     conda install –c conda-forge keras
     
   - It can also be installed using pip as follows:\
     pip install --upgrade pip\
     pip install tensorflow
     
     
2. Run on Google Colaboratory (CoLab).
   - No installation required.
   - The images and data used in code has to be uploaded (in the files area) afresh everytime   CoLab is used.
   - One trick around is to place all the data and images in Google Drive and mount the drive in Google Colab as shown in the following code. This will save us from uploading the images again anad again everytime use use the notebook.
     

# Building a CNN Model

## Loading Data

- We will train our model on CIFAR-10 dataset.
- The CIFAR-10 dataset consists of 60000 32x32 color images in 10 classes, with 6000 images per class.
- There are 50000 training images and 10000 test images.
- The dataset can also be downloaded from
https://www.cs.toronto.edu/~kriz/cifar.html

Categories in CIFAR10
- Category 0: airplane
- Category 1: automobile
- Category 2: bird
- Category 3: cat
- Category 4: deer
- Category 5: dog
- Category 6: frog
- Category 7: horse
- Category 8: ship
- Category 9: truck


In [ ]:
(x_train_org, y_train_org), (x_test_org, y_test_org) = cifar10.load_data()

In [ ]:
%matplotlib inline
fig = plt.figure(figsize=(20,5))
for i in range(36):
    ax = fig.add_subplot(3, 12, i + 1, xticks=[], yticks=[])
    ax.imshow(np.squeeze(x_train_org[i]))

## Preprocessing Data

### Scaling the train and test data

In [ ]:
#Scaling the training data between 0 and 1
x_train_scaled = x_train_org / 255.0

In [ ]:
#Scaling the test tensor between 0 and 1.
x_test_scaled = x_test_org / 255.0

In [ ]:
#Renaming x_test_scaled_flat to simply x_test for easier use later.
x_test = x_test_scaled
#Renaming y_test_org to simply y_test for easier use later.
y_test=y_test_org

### Creating Validation Dataset
Since the dataset is pretty large, let us make a third set also; by dividing the trainset further into train and validation sets.

In [ ]:
VALIDATION_SIZE=10000

##Creating validation set
x_val = x_train_scaled[:VALIDATION_SIZE]
y_val = y_train_org[:VALIDATION_SIZE]
x_val.shape

In [ ]:
##Creating the remaining train set
x_train = x_train_scaled[VALIDATION_SIZE:]
y_train = y_train_org[VALIDATION_SIZE:]
x_train.shape

So now we have three scaled and flattened datasets:
- The train set (x_train,y_train) having 40000 samples
- The validation set (x_val,y_val) having 10000 samples
- The test set (x_test,y_test) having 10000 samples

## Define the Convolutional Neural Network using Keras

In [ ]:
model = Sequential([
    Input(shape=(32,32,3)),
    Conv2D(filters=16, kernel_size=2, padding='same', activation='relu'),
    MaxPooling2D(pool_size=2),
    Conv2D(filters=32, kernel_size=2, padding='same', activation='relu'),
    MaxPooling2D(pool_size=2),
    Conv2D(filters=64, kernel_size=2, padding='same', activation='relu'),
    MaxPooling2D(pool_size=2),
    Dropout(0.3,seed=42,),
    Flatten(), #Flattens the last feature map into a vector of features
    Dense(500, activation='relu'),
    Dropout(0.4,seed=42,),
    Dense(10, activation='softmax')])
#pool_size=2 in the pooling layets sets both kernel_size and strides to 2.
#In conv2D, the default for strides is 1.

In [ ]:
model.summary()

## Fitting the Model

In [ ]:
#Setting some hyperparameters
batch_size = 256
nr_epochs = 150

In [ ]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam',
              metrics=['accuracy'])

In [ ]:
%%time
#Training the model
history = model.fit(x_train, y_train, batch_size=batch_size, epochs=nr_epochs,
                    verbose=1, validation_data=(x_val, y_val))

### Some points to remember:

- Models starts training from the point where it left last time. To avoid this, compile the model afresh.

- On different runs, the optimiser may start from a different point (different initial values of the weights), hence generating a different graphs each time.

In [ ]:
#Plotting the learning curves for train and test accuracy
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.legend()
plt.show()

In [ ]:
#Plotting the learning curves for train and test loss
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.legend()
plt.show()

## Model Evaluation
- Now let us evaluate the model for the test dataset that we initially kept aside.

In [ ]:
#Recalling the metrics that we set during compilation of the model.
model.metrics_names

In [ ]:
#Let us print the loss function value and overall accuracy of our model on train as well as test data.
train_loss, train_accuracy = model.evaluate(x_train, y_train)
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f'Train loss is {train_loss:0.3} and train accuracy is {train_accuracy:0.1%}')
print(f'Test loss is {test_loss:0.3} and test accuracy is {test_accuracy:0.1%}')

### Confusion Matrix
- Now let us print the confusion matrix and find recall and precision values by using the formulas that we studied in class previously.

In [ ]:
predictions=np.argmax(model.predict(x_test), axis=1)
conf_matrix = confusion_matrix(y_true=y_test, y_pred=predictions)

In [ ]:
conf_matrix

In [ ]:
# displaying confustion matrix
plt.figure(figsize=(7,7), dpi=95)
plt.imshow(conf_matrix, cmap=plt.cm.Greens)

plt.title('Confusion Matrix', fontsize=16)
plt.ylabel('Actual Labels', fontsize=12)
plt.xlabel('Predicted Labels', fontsize=12)

tick_marks = np.arange(10)
plt.yticks(tick_marks)
plt.xticks(tick_marks)

plt.colorbar()

for i in range(10):
    for j in range(10):
        plt.text(j, i, conf_matrix[i, j], horizontalalignment='center',
                 color='white' if conf_matrix[i, j] > conf_matrix.max()/2 else 'black')

plt.show()

In [ ]:
recall = np.diag(conf_matrix) / np.sum(conf_matrix, axis=1)
recall

In [ ]:
avg_recall = np.mean(recall)
print(f'Model 1 recall score is {avg_recall:.2%}')

In [ ]:
precision = np.diag(conf_matrix) / np.sum(conf_matrix, axis=0)
precision

In [ ]:
avg_precision = np.mean(precision)
print(f'Model 1 precision score is {avg_precision:.2%}')

f1_score = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall)
print(f'Model 1 f1 score is {f1_score:.2%}')